# SimWorld Spain: Geometry, Meshing, and CFD Workflow

This notebook builds a python-driven parametric heat-exchanger geometry
 - Updates the geometry based on input parameters (pyGeometry)
 - Generates a mesh in Fluent Meshing (pyFluent -> Meshing)
 - Solves the flow/thermal model in Fluent Solver (pyFluent -> Solver)
 - Extracts and visualizes the results (pyFluent Visualization)

In [ ]:

from IPython.display import Image, display, IFrame

# Display an image from file path
display(Image(filename='D:\\AFT\\PyFluent\\PyGeometry\\SimWorld_Spain\\Demo\\header.png'))

### Import modules

In [ ]:
import os
from pathlib import Path

In [ ]:
from ansys.geometry.core import launch_modeler
from ansys.geometry.core.parameters.parameter import ParameterUpdateStatus
from ansys.geometry.core.designer.design import DesignFileFormat
from ansys.geometry.core.misc import UNITS

In [ ]:
import ansys.fluent.core as pyfluent
import ansys.fluent.visualization as pyvisualization

### Helper Utilities

Helper functions for parameter update, mesh export and interactive PyVista rendering.

In [ ]:

import shutil
import tempfile
from pathlib import Path

import pyvista as pv


def render_state(
    session,
    opacity=0.35,
    clip_planes=None,
    temp_dir=None,
    stl_name="state_mesh.stl",
    show_edges=False,
    backend="client",
    notebook=True,
    cleanup=True,
):
    """
    Export current Fluent state to STL, read with PyVista, and render with optional clip planes.

    Parameters
    ----------
    session : ansys.fluent.core.session.Session
        Active PyFluent session.
    opacity : float, optional
        Opacity for rendered mesh or clip-plane actors.
    clip_planes : list[dict], optional
        Each dict can define:
        - normal (e.g. "x", "-z")
        - origin (x, y, z) tuple
        - show_edges (bool, optional)
    temp_dir : str or Path, optional
        Folder where STL is exported. If None, a temporary folder is created.
    stl_name : str, optional
        STL file name to export.
    show_edges : bool, optional
        Default edge rendering for actors.
    backend : str, optional
        PyVista Jupyter backend.
    notebook : bool, optional
        Use notebook plotter mode.
    cleanup : bool, optional
        Remove auto-created temp folder after plotting.
    """
    created_temp_dir = False
    if temp_dir is None:
        temp_dir = Path(tempfile.mkdtemp(prefix="pyfluent_stl_"))
        created_temp_dir = True
    else:
        temp_dir = Path(temp_dir)
        temp_dir.mkdir(parents=True, exist_ok=True)

    stl_path = temp_dir / stl_name

    try:
        if stl_path.exists():
            stl_path.unlink()
        session.tui.file.export.stl(str(stl_path))
        mesh = pv.read(str(stl_path))

        pv.set_jupyter_backend(backend)
        plotter = pv.Plotter(notebook=notebook)

        clipped = mesh
        if clip_planes:
            for plane in clip_planes:
                plane_kwargs = {
                    "normal": plane["normal"],
                    "origin": tuple(plane["origin"])
                }
                clipped = clipped.clip(**plane_kwargs)
        plotter.add_mesh(clipped, show_edges=show_edges, opacity=opacity)

        plotter.show()
        final_stl_path = None if (cleanup and created_temp_dir) else stl_path
        return plotter, mesh, final_stl_path
    finally:
        if cleanup and created_temp_dir:
            shutil.rmtree(temp_dir, ignore_errors=True)

In [ ]:

def update_parameters(design, design_point):
    
    parameters = design.get_all_parameters()

    updates = [param for param in parameters]

    results = []

    for param in updates:
        if param.name in design_point:
            param.dimension_value = design_point[param.name]
            results.append(design.set_parameter(param))

    success = all(r == ParameterUpdateStatus.SUCCESS for r in results)

    if success:
        print("Parameter update SUCCESS")
    else:
        print("Parameter update FAILED", results)

    return success

## Geometry Inputs and Parameters

Define the geometry file paths and key design/physics parameters used to update the CAD model before meshing.

In [ ]:
GEOMETRY_FILE = Path(r"D:\AFT\PyFluent\PyGeometry\SimWorld_Spain\Demo\heat_exchanger.dsco")
MESH_FILE = GEOMETRY_FILE.parent / "heat_exchanger.msh.h5"
CASE_FILE = GEOMETRY_FILE.parent / "heat_exchanger.cas.h5"

In [19]:
fin_height = 20.0 * UNITS.mm
fin_thick = 3.0 * UNITS.mm
fin_number = 5
fin_space = (46.0 * UNITS.mm - fin_thick) / (fin_number - 1)
angle = 45.0 * UNITS.degree

number_of_boundary_layers = 5

heat_flux = 100000.0

## Update geometry 

In [20]:
input_parameters = {
    "fin_height": fin_height,
    "fin_thick": fin_thick,
    "fin_space": fin_space,
    "fin_number": fin_number,
    "angle": angle,
}


modeler = launch_modeler()
design = modeler.open_file(GEOMETRY_FILE, upload_to_server=False)
fin_number = input_parameters.pop('fin_number')
update_parameters(design, input_parameters)
update_parameters(design, {'fin_number': fin_number})

design.download(GEOMETRY_FILE.parent / "heat_exchanger.pmdb", DesignFileFormat.PMDB)
design.close()


WARNING -  -  launcher - _launch_with_automatic_detection - The local Docker container could not be started. Trying to start the Geometry service locally.
WARNING - localhost:63184 -  design - __export_and_download - Failed to download the file in PMDB format. Attempting to stream download.


Parameter update SUCCESS
Parameter update SUCCESS


## Fluent Meshing Workflow

### Launch fluent in meshing mode

In [21]:
session = pyfluent.launch_fluent(
    product_version=pyfluent.FluentVersion.v271,
    dimension=pyfluent.Dimension.THREE,
    mode=pyfluent.FluentMode.MESHING,
    precision=pyfluent.Precision.DOUBLE,
    processor_count=4,
    ui_mode=pyfluent.UIMode.NO_GUI
)

### Start watertight workflow

In [22]:
wt = session.watertight()

D:\AFT\PyFluent\PyGeometry\venv_geom\Lib\site-packages\ansys\fluent\core\pyfluent_warnings.py:63: FluentDevVersionWarning: ⚠️ Warning: You are using PyFluent with an unreleased or development version of Fluent.
Compatibility is not guaranteed, and unexpected behavior may occur. Please use a released version of Fluent that is officially supported by this version of PyFluent.
  warnings.warn(


### Import geometry

In [23]:
import_geometry = wt.import_geometry
import_geometry.arguments(file_name=str(GEOMETRY_FILE.parent / "heat_exchanger.pmdb"))
import_geometry()



Importing one geom object per program-controlled and one zone per body ...
    C:\\Program Files\\ANSYS Inc\\v271\\commonfiles\\CPython\\3_10\\winx64\\Release\\python\\..\\Ansys\\TGrid\\CADReaders.py started by paguado on SNPS-MRN82cUVFv winx64 on Wed May 27 20:42:53 2026
    using Python 3.10.19 (remotes/origin/fa2e225dce21d1c42e354b4b60f08c5fdd382a28-dirty:fa2e225d, Mar 31 2) [MSC v.1942 64 bit (AMD64)]
    
    using Ansys.Meshing.FieldMesher build Apr 27 2026 18:02:17
    
    running ANSYS TGrid CADToTGridConverter ...
    setting up parameters ...
    setting up parameters done.
    running conversion ...
    converting 1 file(s) from Workbench to FLTG using output path 'D:\\AFT\\PyFluent\\PyGeometry\\SimWorld_Spain\\Demo\\FM_SNPS-MRN82cUVFv_6352/out1779907372.3379936352.tgf'
    converting file 'heat_exchanger.pmdb' (1 of 1) from Workbench to FLTG using output path 'D:\\AFT\\PyFluent\\PyGeometry\\SimWorld_Spain\\Demo\\FM_SNPS-MRN82cUVFv_6352'
    importing data ...
    importi

True

### Visualize geometry

In [24]:
clip_planes = [
    {"normal": "-x", "origin": (-75.0, 0.0, 0.0)},
    {"normal": "x", "origin": (240.0, 0.0, 0.0)},
    {"normal": "z", "origin": (0.0, 0.0, 55.0)},
]
temp_dir = Path(GEOMETRY_FILE).parent / "tmp"
plotter, mesh, stl_path = render_state(
    temp_dir=str(temp_dir),
    session=session,
    opacity=1.0,
    clip_planes=clip_planes,
    show_edges=True
)


Writing "D:\AFT\PyFluent\PyGeometry\SimWorld_Spain\Demo\tmp\state_mesh.stl"...

Done.



Widget(value='<iframe id="pyvista-jupyter_trame__template_P_0x1ca2c671730_0" src="http://localhost:8888/trame-…

### Add local sizing

In [25]:
add_local_sizing = wt.add_local_sizing_wtm
properties = {
 'boi_execution': 'Proximity',
 'boi_cells_per_gap': 5.0,
 'boi_scope_to': 'edges',
 'boi_face_label_list': ['fins'],
 'boi_zoneor_label': 'label',
 'boi_control_name': 'proximity_fins1',
 'boi_min_size': 0.01,
 'add_child': True,
 'boi_growth_rate': 1.2}
add_local_sizing.arguments(**properties)
add_local_sizing.add_child_and_update()


---------------- The Global Min size was adjusted to 0.01 

---------------- 5 cells per gap was added to the edges of proximity_fins1


True

### Generate the surface mesh

In [26]:
generate_surface_mesh = wt.create_surface_mesh
properties = {
    'curvature_normal_angle': 18.0,
    'size_functions': 'Curvature & Proximity',
    'cells_per_gap': 4.0    
}
generate_surface_mesh.cfd_surface_mesh_controls.setState(**properties)
generate_surface_mesh()

Writing "D:\AFT\PyFluent\PyGeometry\SimWorld_Spain\Demo\FM_SNPS-MRN82cUVFv_6352\TaskObject3.msh.h5" ...
writing 2 node zones
writing 14 edge zones 
writing 8 face zones 
writing node curvature data
done.    processing size functions/scoped sizing to create Size Field...

Writing "D:\AFT\PyFluent\PyGeometry\SimWorld_Spain\Demo\FM_SNPS-MRN82cUVFv_6352\heat_exchanger.sf"...

Done.


Importing one mesh object per program-controlled and one zone per body ...
    C:\\Program Files\\ANSYS Inc\\v271\\commonfiles\\CPython\\3_10\\winx64\\Release\\python\\..\\Ansys\\TGrid\\CADReaders.py started by paguado on SNPS-MRN82cUVFv winx64 on Wed May 27 20:43:15 2026
    using Python 3.10.19 (remotes/origin/fa2e225dce21d1c42e354b4b60f08c5fdd382a28-dirty:fa2e225d, Mar 31 2) [MSC v.1942 64 bit (AMD64)]
    
    using Ansys.Meshing.FieldMesher build Apr 27 2026 18:02:17
    
    running ANSYS TGrid CADToTGridConverter ...
    setting up parameters ...
    setting up parameters done.
    running conversion ..

True

### Visualize the surface mesh

In [27]:
clip_planes = [
    {"normal": "-x", "origin": (-75.0, 0.0, 0.0)},
    {"normal": "x", "origin": (240.0, 0.0, 0.0)},
    {"normal": "z", "origin": (0.0, 0.0, 55.0)},
]
temp_dir = Path(GEOMETRY_FILE).parent / "tmp"
plotter, mesh, stl_path = render_state(
    temp_dir=str(temp_dir),
    session=session,
    opacity=1.0,
    clip_planes=clip_planes,
    show_edges=True
)


Writing "D:\AFT\PyFluent\PyGeometry\SimWorld_Spain\Demo\tmp\state_mesh.stl"...

Done.



Widget(value='<iframe id="pyvista-jupyter_trame__template_P_0x1ca2c656c60_1" src="http://localhost:8888/trame-…

### Apply shared topology and update boundaries

In [28]:
describe_geometry = wt.describe_geometry
properties = {
 'non_conformal': False,
 'capping_required': False,
 'invoke_share_topology': 'Yes',
 'wall_to_internal': False,
 'multizone': False,
 'setup_type': 'fluid_solid_voids'}
describe_geometry.arguments()
describe_geometry()

wt.apply_share_topology()
wt.update_boundaries()
wt.create_regions()
wt.update_regions()


---------------- Describe Geometry task complete in  0.00 minutes.
Writing "D:\AFT\PyFluent\PyGeometry\SimWorld_Spain\Demo\FM_SNPS-MRN82cUVFv_6352\TaskObject5.msh.h5" ...
writing 3 node zones
writing 26 edge zones 
writing 16 face zones 
writing node curvature data
done.1 pair(s) found.
Joining...
 fluid and hx_source 
Done.
0 pair(s) found.

    computing regions...done
0 faces marked.
Deleting import_curvature_0
Deleting import_proximity_0
    processing size functions/scoped sizing to create Size Field...

Writing "D:\AFT\PyFluent\PyGeometry\SimWorld_Spain\Demo\FM_SNPS-MRN82cUVFv_6352\heat_exchanger.sf"...

Done.


remeshing...


------------------------- --------------------- -------------------- ---------------- ----------
                     name skewed-cells (> 0.80)    averaged-skewness maximum-skewness face count
------------------------- --------------------- -------------------- ---------------- ----------

                    fluid                     0          0.0286177

True

### Add boundary layers

In [29]:
add_boundary_layers = wt.add_boundary_layers
properties = {
    'control_name': 'uniform_bl',
    'rate': 1.2,
    'region_scope': ['fluid'],
    'offset_method_type': 'uniform',
    'face_scope': {
        'grow_on': 'selected-labels',
        'regions_type': 'named-regions'
    },
    'add_child': 'yes',
    'bl_label_list': ['fins', 'hx_side', 'hx_top'],
    'first_height': 0.05,
    'number_of_layers': number_of_boundary_layers,
}

add_boundary_layers.arguments(**properties)
add_boundary_layers.add_child_and_update()


Created Scoped Prism: uniform_bl

---------------- Inflation control added to heat_exchanger


True

### Create volume mesh

In [30]:
create_volume_mesh = wt.create_volume_mesh_wtm
properties = {
'volume_fill_controls': {
    'tet_poly_max_cell_length': 10.0,
    'growth_rate': 1.2
},
'parallel_meshing': True,
'volume_fill': 'polyhedra'
}
create_volume_mesh.arguments(**properties)
create_volume_mesh()

Writing "D:\AFT\PyFluent\PyGeometry\SimWorld_Spain\Demo\FM_SNPS-MRN82cUVFv_6352\TaskObject11.msh.h5" ...
writing 13 node zones
writing 37 edge zones 
writing 16 face zones 
writing node curvature data
done.
checking object "heat_exchanger"...
    skipping validating regions of mesh object "heat_exchanger"...done.
auto meshing object heat_exchanger...

processing scoped prisms...
    starting orientation...
done.
    setting prism growth...done.
done.
Identifying Topology...

Generating Prisms...

Generating initial mesh...

Refining mesh...

Create polyhedra ...

Merging Domains...
done.

                     name       id cells (quality < 0.05)  minimum quality cell count
------------------------- -------- ---------------------- ---------------- ----------
           heat_exchanger     4709                      0         0.331026      50422
                    fluid     4706                      0       0.20783115     226131

                     name       id cells (quality < 0.05)  

True

### Save mesh and exit

In [31]:
session.meshing.File.WriteMesh(MESH_FILE)

Writing "heat_exchanger.msh.h5" ...
writing 13 node zones
writing 25 edge zones 
writing 10 face zones 
writing 2 cell zones 
writing boundary layer flags
writing node curvature data
done.Copying the required intermediate mesh files into heat_exchanger_workflow_files
Done.


True

In [32]:
session.exit()

## Fluent Solver Setup

### Start Fluent in solver mode

In [33]:
solver = pyfluent.launch_fluent(
    product_version=pyfluent.FluentVersion.v271,
    dimension=pyfluent.Dimension.THREE,
    mode=pyfluent.FluentMode.SOLVER,
    precision=pyfluent.Precision.DOUBLE,
    processor_count=16,
    ui_mode=pyfluent.UIMode.NO_GUI
)

In [34]:
settings = solver.settings

D:\AFT\PyFluent\PyGeometry\venv_geom\Lib\site-packages\ansys\fluent\core\pyfluent_warnings.py:63: FluentDevVersionWarning: ⚠️ Warning: You are using PyFluent with an unreleased or development version of Fluent.
Compatibility is not guaranteed, and unexpected behavior may occur. Please use a released version of Fluent that is officially supported by this version of PyFluent.
  warnings.warn(


### Read mesh

In [35]:
settings.file.read_mesh(file_name=MESH_FILE)

Fast-loading "C:\PROGRA~1\ANSYSI~1\v271\fluent\fluent27.1.0\\addons\afd\lib\hdfio.bin"
Done.

Reading from SNPS-MRN82cUVFv:"D:\AFT\PyFluent\PyGeometry\SimWorld_Spain\Demo\heat_exchanger.msh.h5" in NODE0 mode ...
  Reading mesh ...
         will auto partition.
      276553 cells,     2 cell zones ...
         226131 polyhedra cells,  zone id: 4706
          50422 polyhedra cells,  zone id: 4709
     1534078 faces,    10 face zones ...
         311754 polygonal interior faces,  zone id: 4708
        1190012 polygonal interior faces,  zone id: 4705
           2503 polygonal wall faces,  zone id: 3615
           4737 polygonal wall faces,  zone id: 47
           1928 polygonal wall faces,  zone id: 46
          16736 polygonal wall faces,  zone id: 44
           1757 polygonal wall faces,  zone id: 43
            197 polygonal wall faces,  zone id: 42
            197 polygonal wall faces,  zone id: 41
           4257 polygonal wall faces,  zone id: 40
     1125510 nodes,    13 node zones 

### Define models

In [36]:
models = settings.setup.models
models.energy.enabled = True
models.viscous.k_omega_model = 'sst'

### Setup materials

In [37]:
materials = settings.setup.materials
materials.database.copy_materials(
    copy_by_formula = False, 
    names = ['water-liquid'], 
    type = 'fluid'
)
materials.database.copy_materials(
    copy_by_formula = False, 
    names = ['copper'], type = 'solid'
)

D:\AFT\PyFluent\PyGeometry\venv_geom\Lib\site-packages\ansys\fluent\core\solver\flobject.py:1734: PyFluentUserWarning: Unknown keyword 'copy_by_formula' for command '<session>.settings.setup.materials.database.copy_materials'. It will be ignored.
  warnings.warn(



Material 'water-liquid' copied from 'fluent-database' database.


Material 'copper' copied from 'fluent-database' database.



In [38]:
cz = settings.setup.cell_zone_conditions
cz.set_zone_type(new_type = 'fluid', zone_list = ['fluid'])
cz.fluid['fluid'].general.material = 'water-liquid'
cz.solid['heat_exchanger'].general.material = 'copper'

### Setup boundary conditions

In [39]:
bc = settings.setup.boundary_conditions
bc.set_zone_type(new_type = 'velocity-inlet', zone_list = ['inlet'])
bc.velocity_inlet['inlet'].momentum.velocity_magnitude.value = 1
bc.velocity_inlet['inlet'].thermal.temperature.value = 300.0

bc.set_zone_type(new_type = 'pressure-outlet', zone_list = ['outlet'])

for wall in bc.wall:
    if 'source' in wall:
        bc.wall[wall].thermal.heat_flux.value = heat_flux


### Setup numerics

In [40]:
methods = settings.solution.methods
methods.p_v_coupling.flow_scheme = 'SIMPLEC'
methods.spatial_discretization.discretization_scheme['pressure'] = 'presto!'

### Define reports

In [41]:
report_definitions = settings.solution.report_definitions
report_definitions.flux.create()
report_definitions.flux.rename(new = 'mass_flow', old = 'report-def-0')
report_definitions.flux['mass_flow'].boundaries = ['outlet', 'inlet']

monitor = settings.solution.monitor
monitor.report_plots.create()
monitor.report_plots['report-plot-1'].report_defs = ['mass_flow']
monitor.report_plots.rename(new = 'mass_flow_plot', old = 'report-plot-1')

### Run calculation

In [42]:
solution = settings.solution
monitor.residual.options.criterion_type = 'none'
solution.initialization.initialization_type = 'standard'
solution.run_calculation.parameters.iter_count = 100
solution.run_calculation.calculate()


 turbulent viscosity limited to viscosity ratio of 1.000000e+05 in 226131 cells 

  iter  continuity  x-velocity  y-velocity  z-velocity      energy           k       omega     time/iter

 turbulent viscosity limited to viscosity ratio of 1.000000e+05 in 226131 cells 

 turbulent viscosity limited to viscosity ratio of 1.000000e+05 in 18 cells 
     1  1.0000e+00  9.2622e+02  2.6730e-04  2.9645e-04  6.0932e-10  6.3756e-04  1.9087e+04  0:01:39   99

 turbulent viscosity limited to viscosity ratio of 1.000000e+05 in 24 cells 
     2  1.0000e+00  5.8949e-02  1.6295e-03  1.0731e-03  5.0333e-08  2.7925e-02  1.9398e-01  0:01:50   98

 turbulent viscosity limited to viscosity ratio of 1.000000e+05 in 7 cells 

 Reversed flow on 1 face (0.2% area) of pressure-outlet 42.
     3  9.4967e-01  4.1247e-02  2.6280e-03  2.3239e-03  7.5545e-07  1.7088e-02  1.5956e-01  0:01:39   97

 turbulent viscosity limited to viscosity ratio of 1.000000e+05 in 10 cells 

 Reversed flow on 8 faces (3.1% area) of p

### Create post-processing surfaces

In [43]:
surfaces = settings.results.surfaces
surfaces.plane_surface.create(name='plane_xy')
surfaces.plane_surface['plane_xy'].method = 'xy-plane'
surfaces.plane_surface['plane_xy'].z = 0.02
surfaces.plane_surface.create(name='plane_yz')
surfaces.plane_surface['plane_yz'].method = 'yz-plane'
surfaces.plane_surface['plane_yz'].x = 0.0
surfaces.plane_surface.create(name='plane_xz')
surfaces.plane_surface['plane_xz'].method = 'zx-plane'

In [44]:
surf_planes = ['plane_xz', 'plane_xy', 'plane_yz']
surf_hx = []
for wall in bc.wall:
    if 'heat_exchanger' in wall and 'shadow' not in wall:
        surf_hx.append(wall)

## Post-Processing and Visualization

Create analysis surfaces and display mesh, contours, and pathlines using Fluent visualization objects and PyVista backend.

In [45]:
from ansys.fluent.visualization import Surface, Mesh, Contour, Pathline
from ansys.fluent.visualization import GraphicsWindow

In [46]:

graphics = settings.results.graphics
graphics.mesh.create(name = 'mesh-yz')
graphics.mesh['mesh-yz'].surfaces_list = ['plane_yz']
graphics.mesh['mesh-yz'].options.edges = True

graphics.contour.create(name='contour_temperature')
graphics.contour['contour_temperature'].field = 'temperature'

graphics.contour['contour_temperature'].surfaces_list = surf_planes + surf_hx

graphics.contour.create(name='contour_velocity')
graphics.contour['contour_velocity'].field = 'velocity-magnitude'
graphics.contour['contour_velocity'].surfaces_list = surf_planes


In [47]:

graphics.pathline.create(name='pathlines-temp')
pathline = graphics.pathline['pathlines-temp']
pathline.field = 'temperature'
pathline.release_from_surfaces = ['inlet']
pathline.range_options.compute()
pathline.release_from_surfaces = surf_hx
pathline.option.skip = 5


### Show mesh in yz plane

In [48]:
pv.set_jupyter_backend('client')
mesh = Mesh(
    solver=solver, show_edges=True, surfaces=["plane_yz"]
)
window = GraphicsWindow()
window.add_graphics(mesh)
window.show()

Widget(value='<iframe id="pyvista-jupyter_trame__template_P_0x1ca2fb962a0_2" src="http://localhost:8888/trame-…

### Show temperature contour

In [49]:
temperature_contour_object = Contour(
    solver=solver, field="temperature", surfaces=surf_hx
)
window = GraphicsWindow()
window.add_graphics(temperature_contour_object)
window.show()

Widget(value='<iframe id="pyvista-jupyter_trame__template_P_0x1ca2fb97da0_3" src="http://localhost:8888/trame-…

### Show velocity contour

In [50]:
velocity_contour_object = Contour(
    solver=solver, field="velocity-magnitude", surfaces=surf_planes
)
window = GraphicsWindow()
window.add_graphics(velocity_contour_object)
window.show()

Widget(value='<iframe id="pyvista-jupyter_trame__template_P_0x1ca2c9d3110_4" src="http://localhost:8888/trame-…

### Show pathlines

In [51]:
surf_hx = []
for wall in bc.wall:
    if 'heat_exchanger' in wall and 'shadow' not in wall:
        surf_hx.append(wall)

In [52]:
mesh = Mesh(
    solver=solver, show_edges=True, surfaces=surf_hx
)
window = GraphicsWindow()
window.add_graphics(mesh)

pathlines = Pathline(
    solver=solver,
    field="temperature",
    surfaces = surf_hx,
    skip = 20
)
window = GraphicsWindow()
window.add_graphics(mesh)
window.add_graphics(pathlines)
window.show()



number tracked = 797, escaped = 785, aborted = 1, incomplete = 11

number tracked = 92, escaped = 91, incomplete = 1

number tracked = 226, escaped = 209, incomplete = 17


Widget(value='<iframe id="pyvista-jupyter_trame__template_P_0x1ca3001c140_5" src="http://localhost:8888/trame-…

## Export HTML report

In [53]:
sim_reports = solver.settings.results.report.simulation_reports
sim_reports.generate_simulation_report()
REPORT_FOLDER = CASE_FILE.parent / "heat_exchanger"
sim_reports.export_simulation_report_as_html(report_name="Ansys Fluent Simulation Report", output_dir=REPORT_FOLDER)

Connecting To Ansys Dynamic Reporting
Adding Data
Adding Plots
Adding Images

number tracked = 3901, escaped = 3721, incomplete = 180
Managing Templates
Report Finished
Provided report name is not allowed. Replacing with a unique report name

Report Address:
http://127.0.0.1:2384/reports/report_display/?view=c8de57c8-59fc-11f1-848e-ac91a12605cd&colormode=dark

Standalone HTML report created in D:/AFT/PyFluent/PyGeometry/SimWorld_Spain/Demo/heat_exchanger


In [54]:
solver.exit()